# 07 ML 回测分析

完整闭环：加载训练好的模型 → 生成全历史预测信号 → 接入 BacktestEngine → 绩效分析。

---

**流程：**
1. 加载配置 & 模型
2. 生成预测信号 & 信号分布
3. 运行回测
4. 权益曲线 & 回撤
5. 成交标注
6. 月度收益热图
7. 滚动 Sharpe

In [ ]:
import syssys.path.insert(0, '..')import numpy as npimport pandas as pdimport plotly.graph_objects as goimport plotly.express as pxfrom plotly.subplots import make_subplotsfrom pathlib import Pathfrom ml.train import load_configfrom ml.backtest import run_ml_backtestfrom futuresquant.backtest.analyzer import PerformanceAnalyzerCFG_PATH = Path('..') / 'ml' / 'config.yaml'cfg = load_config(CFG_PATH)print(f'品种: {cfg["data"]["product"]}')print(f'频率: {cfg["data"].get("freq", "1D")}')print(f'标签: {cfg["labels"]["label_type"]}  forward_bars={cfg["labels"].get("forward_bars", 1)}')

## 1. 运行 ML 回测

In [ ]:
# 可指定模型路径，None 则自动取最新MODEL_PATH = None  # e.g. Path('..') / 'ml' / 'artifacts' / 'model_xxx.pkl'ENTRY_Q = 0.8     # 开仓信号分位EXIT_Q  = 0.5     # 平仓信号分位ROLLING = 500     # 滚动窗口result, signal = run_ml_backtest(    cfg,    model_path=MODEL_PATH,    entry_quantile=ENTRY_Q,    exit_quantile=EXIT_Q,    rolling_window=ROLLING,)

## 2. 预测信号分布

In [ ]:
fig = make_subplots(rows=2, cols=1, subplot_titles=['信号时序', '信号分布'],                    row_heights=[0.6, 0.4], vertical_spacing=0.1)fig.add_trace(go.Scatter(x=signal.index, y=signal.values,                         mode='lines', name='ML 信号',                         line=dict(width=1, color='steelblue')),              row=1, col=1)fig.add_hline(y=0, line_color='black', line_width=0.5, row=1, col=1)fig.add_trace(go.Histogram(x=signal.dropna().values, nbinsx=80,                           name='分布', marker_color='steelblue', opacity=0.7),              row=2, col=1)fig.update_layout(title='ML 预测信号', height=550, showlegend=False)fig.show()print(f'\n信号统计:')print(signal.describe().round(6).to_string())

## 3. 权益曲线 & 回撤

In [ ]:
equity = result.account.equity_curve()analyzer = PerformanceAnalyzer(equity, result.config.initial_capital)drawdown = analyzer.drawdown_series()fig = make_subplots(rows=3, cols=1, shared_xaxes=True,                    subplot_titles=['权益曲线', '回撤 (%)', '标的价格'],                    row_heights=[0.5, 0.25, 0.25], vertical_spacing=0.06)fig.add_trace(go.Scatter(x=equity.index, y=equity.values,                         name='权益', line=dict(color='#1f77b4')), row=1, col=1)fig.add_hline(y=result.config.initial_capital, line_dash='dash',              line_color='gray', row=1, col=1)fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values * 100,                         name='回撤', fill='tozeroy',                         line=dict(color='#d62728'),                         fillcolor='rgba(214,39,40,0.2)'), row=2, col=1)klines = result.klinesprice_5min = klines['close'].resample('5min').last().dropna()fig.add_trace(go.Scatter(x=price_5min.index, y=price_5min.values,                         name='收盘价', line=dict(color='#2ca02c', width=1)), row=3, col=1)fig.update_layout(title='ML 策略回测', height=700, hovermode='x unified')fig.update_yaxes(title_text='元', row=1, col=1)fig.update_yaxes(title_text='%', row=2, col=1)fig.show()

## 4. 成交标注

In [ ]:
from futuresquant.backtest.order import Direction, Offsetfills = result.account.fillsfills_df = pd.DataFrame([    {'time': f.time, 'direction': f.direction.value,     'offset': f.offset.value, 'price': f.price,     'volume': f.volume, 'commission': f.commission}    for f in fills])print(f'总成交笔数: {len(fills_df)}')print(f'总手续费: {fills_df["commission"].sum():.2f} 元')if len(fills_df) > 0:    opens = fills_df[fills_df['offset'] == 'OPEN']    buy_opens = opens[opens['direction'] == 'LONG']    sell_opens = opens[opens['direction'] == 'SHORT']    fig = go.Figure()    fig.add_trace(go.Scatter(x=price_5min.index, y=price_5min.values,                             name='价格(5min)', line=dict(color='gray', width=1)))    fig.add_trace(go.Scatter(        x=buy_opens['time'].iloc[:200], y=buy_opens['price'].iloc[:200],        mode='markers', name='买入开仓',        marker=dict(symbol='triangle-up', size=8, color='#2ca02c')))    fig.add_trace(go.Scatter(        x=sell_opens['time'].iloc[:200], y=sell_opens['price'].iloc[:200],        mode='markers', name='卖出开仓',        marker=dict(symbol='triangle-down', size=8, color='#d62728')))    fig.update_layout(title='成交点标注（前200笔开仓）', height=450)    fig.show()

## 5. 月度收益热图

In [ ]:
eq_daily = equity.resample('1D').last().dropna()monthly_ret = eq_daily.resample('ME').last().pct_change().dropna()monthly_df = pd.DataFrame({    'year': monthly_ret.index.year,    'month': monthly_ret.index.month,    'return': monthly_ret.values,})pivot_m = monthly_df.pivot(index='year', columns='month', values='return') * 100month_names = ['Jan','Feb','Mar','Apr','May','Jun',               'Jul','Aug','Sep','Oct','Nov','Dec']pivot_m.columns = [month_names[c-1] for c in pivot_m.columns]fig = px.imshow(pivot_m, text_auto='.1f', color_continuous_scale='RdYlGn',                color_continuous_midpoint=0,                title='月度收益率 (%)', aspect='auto')fig.update_layout(height=300)fig.show()

## 6. 滚动 Sharpe

In [ ]:
rolling_sharpe = analyzer.rolling_sharpe(window_bars=240 * 20).dropna()fig = go.Figure()fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe.values,                         name='20日滚动 Sharpe', line=dict(color='#9467bd')))fig.add_hline(y=0, line_color='black', line_width=0.8)fig.add_hline(y=1, line_dash='dash', line_color='green', annotation_text='Sharpe=1')fig.update_layout(title='策略滚动 Sharpe（20日）', height=380)fig.show()

## 7. 绩效汇总

In [ ]:
m = result.metricsprint('ML 策略回测绩效:')print('=' * 45)for k, v in m.items():    if isinstance(v, float):        print(f'  {k:<25} {v:>12.4f}')    else:        print(f'  {k:<25} {str(v):>12}')print('=' * 45)